## HumanInTheLoopMiddleware中间件
在 工具调用前 中断Agent运行，等待用户对工具调用请求决策。可选的决策有：`approve（同意执行）`、 `edit（编辑调用配置后执行）`、 `reject（拒绝执行）`

### 参数说明
#### interrupt_on —工具名和中断策略的映射
策略可以是True、False或InterruptOnConfig对象，精细控制决策选项。
- True表示所有决策(approve, edit, reject) 都可以选择，
- False表示不中断，即无需审批即可执行。
  
InterruptOnConfig 是一个TypedDict的子类，可以用字典直接赋值。支持的Key有：
① allowed_decisions 精细控制中断后允许的决策。
② description ：特定工具的中断描述信息，优先级高于description_prefix，后 者会更
改 所有 工具中断的描述。

#### 参数2：description_prefix —自定义中断描述
默认为 "Tool execution requires approval" ，下面的举例可以看到效果

In [1]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气
    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "发送邮件中断啦",
                },
            },
            description_prefix="中断啦",
        ),
    ],
)

config = {"configurable": {"thread_id": "1"}}

# 第一次调用：会暂停在发送邮件前
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "请帮我查询今天北京的天气"
                    "查询今日新闻"
                    "查看ID为 'sk2131421' 的邮件内容，"
                    "向15641685664@qq.com发送邮件，标题是'哈哈哈'，"
                    "内容是：'你好啊'"
                    "同时做这四件事"
                )
            )
        ]
    },
    config=config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)

print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)

# print("==== 逐个打印 interrupt 请求 ====")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='ed12fe08-922f-4270-bcd0-084a3c2028b1'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Query 
today\'s weather in Beijing\n   - Query today\'s news\n   - Read email with ID \'sk2131421\'\n   - Send an email to
\'15641685664@qq.com\' with subject \'哈哈哈\' and body \'你好啊\'\n   - Do all four things simultaneously 
("同时做这四件事")\n\n2.  **Identify Required Tools & Parameters:**\n   - `get_weather`: city="北京", 
is_forcast=False (default, user didn\'t specify forecast)\n   - `get_news`: no parameters needed\n   - 
`read_email_tool`: email_id="sk2131421"\n   - `send_email_tool`: recipient="15641685664@qq.com", subject="哈哈哈", 
body="你好啊"\n\n3.  **Check Tool Definitions:**\n   - `get_weather(city, is_forcast)`: `city` required, 
`is_forcast` optional boolean (default false)\n   - `get_news()`: no parameters\n   - `read_email_tool(email_id)`: 
`email_id` required string\n   - `send_email_tool(recipient, subject, body)`: all three required strings\n\n   All 
parameters match the requirements.\n\n4.  **Execute Tool Calls (Mental Simulation):**\n   I will call all four 
tools in parallel as requested.\n   - `get_weather(city="北京", is_forcast=false)`\n   - `get_news()`\n   - 
`read_email_tool(email_id="sk2131421")`\n   - `send_email_tool(recipient="15641685664@qq.com", subject="哈哈哈", 
body="你好啊")`\n\n   I\'ll generate the tool calls now. \n   Note: Since I\'m an AI, I don\'t actually execute 
them in real-time, but I will format the response exactly as the system expects for multi-tool invocation.\n\n5.  
**Construct Response:**\n   I will list the function calls clearly. Then, after receiving the simulated results 
(which I\'ll just present as expected outputs or handle gracefully), I\'ll summarize the results for the user. 
Since this is a direct prompt, I just need to output the tool calls. The system will handle the execution. I\'ll 
just output the tool calls directly.\n\n   Wait, the prompt says "同时做这四件事" (do these four things at the same
time). I should just make the four function calls.\n\n   Let\'s prepare the JSON/structured format for the function
calls.\n   ```json\n   [\n     {"name": "get_weather", "arguments": {"city": "北京", "is_forcast": false}},\n     
{"name": "get_news", "arguments": {}},\n     {"name": "read_email_tool", "arguments": {"email_id": "sk2131421"}},\n
{"name": "send_email_tool", "arguments": {"recipient": "15641685664@qq.com", "subject": "哈哈哈", "body": 
"你好啊"}}\n   ]\n   ```\n   This matches the standard format. I will output it. \n   Self-Correction/Verification 
during thought:\n   - Check parameter names: `is_forcast` (typo in schema? It says `is_forcast`, I\'ll use exactly 
that).\n   - `get_news` has no parameters.\n   - `read_email_tool` needs `email_id`.\n   - `send_email_tool` needs 
`recipient`, `subject`, `body`.\n   All good. I will proceed. \n   Output matches the function calling format.✅\n 
I will generate the tool calls now. \n   *(Self-Correction/Note during generation)*: I should strictly follow the 
tool definition. The schema uses `is_forcast`. I\'ll use `false` as default. Everything else is straightforward.\n 
Proceed. \n   Output: [Function Calls] -> System processes -> Returns results -> I summarize. But since I\'m 
generating the first turn, I just output the calls.\n   Actually, in this environment, I just output the tool calls
directly. The system will handle the rest. I\'ll produce the exact tool calls. \n   Let\'s do it. \n   
`get_weather(city="北京", is_forcast=False)`\n   `get_news()`\n   `read_email_tool(email_id="sk2131421")`\n   
`send_email_tool(recipient="15641685664@qq.co

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_0bb93cbba59446c7ac41cd14)
 Call ID: call_0bb93cbba59446c7ac41cd14
  Args:
    city: 北京
    is_forcast: False
  get_news (call_199b799ac81b42ab91c96e05)
 Call ID: call_199b799ac81b42ab91c96e05
  Args:
  read_email_tool (call_9aa7f6d2f0cb4be299d148d7)
 Call ID: call_9aa7f6d2f0cb4be299d148d7
  Args:
    email_id: sk2131421
  send_email_tool (call_808bafa3da2840749971e217)
 Call ID: call_808bafa3da2840749971e217
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
========== interrupts ==========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='63d42777b1a871731cf9ce33a3d3bf5c'
    )
]

{
    'name': 'get_weather',
    'args': {'city': '北京', 'is_forcast': False},
    'description': "中断啦\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
}

{'name': 'get_news', 'args': {}, 'description': '中断啦\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
    'description': '发送邮件中断啦'
}